# Experimentul 2: robustețea baseline-ului VQC la zgomot depolarizant

**VQC-ul original, raport 1:1, antrenat fără zgomot, evaluat la p = 0, 0.001, 0.01, 0.05.**

## Scop
Vrem să măsurăm cât de bine rezistă baseline-ul din Task 1 când același model (aceleași ponderi) rulează pe un simulator cu zgomot, **înainte** să încercăm orice îmbunătățire (de exemplu un circuit mai simplu). Acesta este punctul 1 obligatoriu din Task 2.

## Ce ținem fix
- configurația aleasă în Experimentul 1: **raport 1:1** = `undersample(X_train, y_train, seed)` oficial, aceleași **5 seeds** `[0, 1, 2, 3, 4]`;
- encoderul (`AngleEncoder`, fitat pe setul 1:1 al fiecărui seed), circuitul (ZZ feature map ×2 + ansatz EfficientSU2-style cu 3 straturi, 32 de parametri, readout pe qubitul 0), optimizer-ul, learning rate-ul, batch size-ul, 30 de epoci, alegerea epocii după AUPRC pe validation;
- **antrenarea rămâne fără zgomot**. Zgomotul apare doar la evaluare. Nu facem noise-aware training aici.

## Modelul de zgomot (exact cel din enunț, `noisy_qnode` din Partea 10)
Circuitul este descompus în porțile native {RX, RY, RZ, Rot, H, CNOT, CZ}, iar după **fiecare poartă**, pe **fiecare qubit** atins de ea, urmează un canal depolarizant de probabilitate p. Simularea este exactă, cu matrici de densitate (`default.mixed`).

## Două etape
1. **Etapa 1 (acum): validation.** Pentru fiecare seed și fiecare p: scoruri pe validation la acel zgomot, pragul care maximizează F1 pe aceleași scoruri, metricile și Spearman ρ față de scorurile fără zgomot. Testul **nu** se transformă și **nu** se scorează.
2. **Etapa 2 (evaluarea finală): test**, cu `RUN_TEST = True`. Rulează `evaluate_under_noise` din notebookul oficial, neschimbat, pe fiecare seed. Se rulează **o singură dată**, după ce nu mai luăm decizii pe baza acestor rezultate.

## Ponderile
Experimentul 1 **nu a salvat ponderile** (`runs.jsonl` are doar metrici, `histories.jsonl` doar loss și AUPRC pe epoci). Le reproducem reantrenând **doar cele 5 rulări 1:1** (nu toate cele 15), cu exact aceleași funcții și seeds, și verificăm că istoricul pe epoci, epoca aleasă și AUPRC-ul coincid cu cele salvate de Experimentul 1. Abia apoi le salvăm în `results/result_exp2/weights/`. La rulările următoare se încarcă de acolo.

## 1. Configurație și imports

**Seeds.** `set_seed` fixează numpy și torch; antrenarea folosește un `torch.Generator` dedicat pentru amestecare (ca în Experimentul 1). Simulatoarele PennyLane rulează cu `shots=None` (valori exacte, fără aleatorietate), inclusiv `default.mixed` pentru zgomot.

`RUN_TEST = False`: etapa 1, doar validation. `RETRAIN_IF_MISSING = True`: dacă lipsesc ponderile salvate ale unui seed, se reproduce antrenarea lui 1:1 (~20–35 s per seed pe laptop, după timpii din Experimentul 1).

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pennylane as qml
import sklearn
import torch
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.datasets import fetch_openml
from sklearn.decomposition import PCA
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# ---- etape ----
RUN_TEST = False              # False: etapa 1, doar validation. True: evaluarea finală pe test (protocolul oficial).
RETRAIN_IF_MISSING = True     # reproduce antrenarea 1:1 a unui seed dacă ponderile lui nu sunt salvate

# ---- protocol comun și VQC-ul original (identic cu notebookul oficial și cu Experimentul 1) ----
SPLIT_SEED = 42               # seed-ul splitului comun 60/20/20
N_QUBITS = 4                  # o componentă PCA per qubit
FEATURE_MAP_REPS = 2          # repetări ale ZZ feature map
ANSATZ_REPS = 3               # (3 + 1) * 4 * 2 = 32 parametri
EPOCHS = 30
BATCH_SIZE = 32
LEARNING_RATE = 0.05
SHOTS_PER_EXECUTION = 1024    # doar pentru cost ledger
NOISE_LEVELS = [0.001, 0.01, 0.05]                                   # probabilitatea depolarizantă per poartă
NATIVE_GATES = {"RX", "RY", "RZ", "Rot", "Hadamard", "CNOT", "CZ"}   # setul de porți al modelului de zgomot
SEEDS = [0, 1, 2, 3, 4]
P_ALL = [0.0, *NOISE_LEVELS]

EXPERIMENT_NAME = "noise_baseline"
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks").is_dir() and (p / "src").is_dir())
RESULTS_DIR = REPO_ROOT / "results" / "result_exp2"   # EXPERIMENT_NAME rămâne doar eticheta din config.json
WEIGHTS_DIR = RESULTS_DIR / "weights"
EXP1_DIR = REPO_ROOT / "results" / "result_exp1" / "test1_full_without_noise"   # Experimentul 1: doar citit, niciodată scris

LEGIT_COLOR, FRAUD_COLOR, SEED_DOT_COLOR = "#2a78d6", "#eb6834", "#8a8f98"
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False})


def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(SPLIT_SEED)
print(f"seeds: {SEEDS}   niveluri de zgomot: {P_ALL}   RUN_TEST: {RUN_TEST}")
print("pennylane", qml.__version__, "| torch", torch.__version__, "| sklearn", sklearn.__version__)
print("rezultate:", RESULTS_DIR.relative_to(REPO_ROOT))

## 2. Datele și splitul comun

Exact ca în Experimentul 1: `data/creditcard.csv` (identic cu OpenML 42175), fără eliminări sau reordonări, split stratificat 60/20/20 cu seed 42, verificat față de `split_indices.npz` dacă fișierul există. În etapa 1 matricea de features a testului este ștearsă din memorie; din test păstrăm doar indicii și etichetele.

In [ ]:
DATA_PATH = REPO_ROOT / "data" / "creditcard.csv"
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    data_source = "data/creditcard.csv"
else:
    df = fetch_openml(data_id=42175, as_frame=True, parser="auto").frame
    data_source = "OpenML 42175"
df["Class"] = df["Class"].astype(int)
assert df.shape == (284807, 31) and "Time" in df.columns, f"forma neașteptată: {df.shape}"
assert df.index.equals(pd.RangeIndex(len(df)))

X = df.drop(columns="Class")
y = df["Class"].to_numpy()
X_trval, X_test, y_trval, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SPLIT_SEED)
X_train, X_val, y_train, y_val = train_test_split(X_trval, y_trval, test_size=0.25, stratify=y_trval, random_state=SPLIT_SEED)
rebuilt = {"train": X_train.index.to_numpy(), "val": X_val.index.to_numpy(), "test": X_test.index.to_numpy()}

SPLIT_PATH = REPO_ROOT / "split_indices.npz"
if SPLIT_PATH.exists():
    saved_split = np.load(SPLIT_PATH)
    if not all(np.array_equal(saved_split[k], rebuilt[k]) for k in rebuilt):
        raise ValueError("split_indices.npz nu corespunde splitului oficial reconstruit")
    split_source = "split_indices.npz (identic cu reconstrucția oficială)"
else:
    split_source = "reconstruit ca în notebookul oficial (split_indices.npz lipsește)"

train_idx, val_idx, test_idx = rebuilt["train"], rebuilt["val"], rebuilt["test"]
n_test_rows, n_test_frauds = len(y_test), int(y_test.sum())
assert not (set(train_idx) & set(val_idx)) and not (set(train_idx) & set(test_idx)) and not (set(val_idx) & set(test_idx))
assert (int(y_train.sum()), int(y_val.sum()), n_test_frauds) == (295, 99, 98) and n_test_rows == 56962
del X_trval, y_trval
if not RUN_TEST:
    del X_test   # etapa 1: testul nu se transformă și nu se scorează

print("split:", split_source)
print(f"train {len(train_idx):,} ({int(y_train.sum())} fraude)   validation {len(val_idx):,} ({int(y_val.sum())})   test {n_test_rows:,} ({n_test_frauds})")
print("matricea testului:", "încărcată (RUN_TEST = True)" if RUN_TEST else "ștearsă din memorie (etapa 1)")

## 3. Funcțiile din notebookul oficial

Copiate din `task1_guided_vqc_fraud_detection.ipynb` cu golurile completate, ca în Experimentul 1: `undersample`, `AngleEncoder` (Partea 4), circuitul (Partea 6), `predict` (Partea 7), `choose_threshold` și `evaluate` (Partea 9), `noisy_qnode`, `noisy_predict`, `noise_locations` (Partea 10) și `evaluate_under_noise` (Partea 11), **fără modificări**. `train_vqc_run` este bucla din Experimentul 1 (= `train_vqc` din Partea 7, cu generator dedicat pentru amestecare), necesară ca să reproducem ponderile.

In [ ]:
def undersample(X_part, y_part, seed):
    rng = np.random.default_rng(seed)
    fraud = np.flatnonzero(y_part == 1)
    legit = rng.choice(np.flatnonzero(y_part == 0), size=len(fraud), replace=False)
    keep = rng.permutation(np.concatenate([fraud, legit]))
    return X_part.iloc[keep], y_part[keep]


class AngleEncoder:
    raw_columns = ["Time", "Amount"]

    def __init__(self, n_components, seed):
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=n_components, random_state=seed)
        self.angles = MinMaxScaler(feature_range=(0, np.pi))

    def _scaled(self, frame):
        out = frame.copy()
        out[self.raw_columns] = self.scaler.transform(frame[self.raw_columns])
        return out.to_numpy()

    def fit(self, frame):
        self.scaler.fit(frame[self.raw_columns])
        self.angles.fit(self.pca.fit_transform(self._scaled(frame)))
        return self

    def transform(self, frame):
        return np.clip(self.angles.transform(self.pca.transform(self._scaled(frame))), 0, np.pi)


def feature_map(x):
    """ZZ feature map. x holds N_QUBITS angles in [0, pi], or a batch of them (shape (..., N_QUBITS))."""
    for _ in range(FEATURE_MAP_REPS):
        for i in range(N_QUBITS):
            qml.Hadamard(wires=i)
            qml.RZ(2 * x[..., i], wires=i)
        for i in range(N_QUBITS):
            for j in range(i + 1, N_QUBITS):
                qml.CNOT(wires=[i, j])
                qml.RZ(2 * (np.pi - x[..., i]) * (np.pi - x[..., j]), wires=j)
                qml.CNOT(wires=[i, j])


def ansatz(weights):
    """EfficientSU2-style ansatz; weights has shape (ANSATZ_REPS + 1, N_QUBITS, 2)."""
    for layer in range(ANSATZ_REPS + 1):
        for i in range(N_QUBITS):
            qml.RY(weights[layer, i, 0], wires=i)
            qml.RZ(weights[layer, i, 1], wires=i)
        if layer < ANSATZ_REPS:
            for i in range(N_QUBITS - 1):
                qml.CNOT(wires=[i, i + 1])


def vqc(x, weights):
    feature_map(x)
    ansatz(weights)
    return qml.expval(qml.PauliZ(0))


device = qml.device("default.qubit", wires=N_QUBITS)   # shots=None: valori de așteptare exacte
circuit = qml.QNode(vqc, device, interface="torch", diff_method="backprop")


def fraud_probability(x, weights):
    return (1 - circuit(x, weights)) / 2


def init_weights(seed):
    generator = torch.Generator().manual_seed(seed)
    return torch.nn.Parameter(2 * np.pi * torch.rand(ANSATZ_REPS + 1, N_QUBITS, 2, generator=generator, dtype=torch.float64))


def predict(Z, weights, model=None, chunk=20000):
    model = model or fraud_probability
    X_t = torch.tensor(Z)
    with torch.no_grad():
        return torch.cat([model(X_t[k:k + chunk], weights) for k in range(0, len(X_t), chunk)]).numpy()


def train_vqc_run(Z_tr, y_tr, Z_valid, y_valid, initial_weights, seed, epochs=EPOCHS):
    set_seed(seed)
    shuffle_generator = torch.Generator().manual_seed(seed)
    weights = torch.nn.Parameter(initial_weights.detach().clone())   # copie independentă
    optimizer = torch.optim.Adam([weights], lr=LEARNING_RATE)
    loss_fn = torch.nn.BCELoss()
    X_t = torch.tensor(Z_tr)
    y_t = torch.tensor(y_tr, dtype=torch.float64)
    history, best_ap, best_weights, best_epoch = [], -1.0, None, None
    start = time.perf_counter()
    for epoch in range(1, epochs + 1):
        order = torch.randperm(len(X_t), generator=shuffle_generator)
        losses = []
        for k in range(0, len(X_t), BATCH_SIZE):
            idx = order[k:k + BATCH_SIZE]
            optimizer.zero_grad()
            loss = loss_fn(fraud_probability(X_t[idx], weights), y_t[idx])
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        val_ap = average_precision_score(y_valid, predict(Z_valid, weights))
        history.append({"epoch": epoch, "train loss": float(np.mean(losses)), "validation AUPRC": float(val_ap)})
        if val_ap > best_ap:
            best_ap, best_weights, best_epoch = val_ap, weights.detach().clone(), epoch
    return {"weights": best_weights, "history": history, "best_epoch": best_epoch, "best_val_ap": float(best_ap),
            "time_train_s": time.perf_counter() - start}


def choose_threshold(y_true, scores):
    """The threshold that maximises F1 on (y_true, scores)."""
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    f1 = 2 * precision * recall / np.clip(precision + recall, 1e-12, None)
    return thresholds[np.argmax(f1[:-1])]


def evaluate(y_true, scores, threshold):
    predicted = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    return {
        "AUPRC": average_precision_score(y_true, scores),
        "AUROC": roc_auc_score(y_true, scores),
        "precision": precision_score(y_true, predicted, zero_division=0),
        "recall": recall_score(y_true, predicted, zero_division=0),
        "F1": f1_score(y_true, predicted, zero_division=0),
        "MCC": matthews_corrcoef(y_true, predicted),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
    }


def noisy_qnode(circuit_fn, p, n_wires=N_QUBITS):
    """QNode of circuit_fn under the challenge's noise model: native gates, then a depolarizing channel of
    probability p after every gate, on each of its qubits. Differentiable with backprop."""
    noisy_device = qml.noise.insert(qml.device("default.mixed", wires=n_wires), qml.DepolarizingChannel, p, position="all")
    qnode = qml.QNode(circuit_fn, noisy_device, interface="torch", diff_method="backprop")
    return qml.transforms.decompose(qnode, gate_set=NATIVE_GATES)


def noisy_predict(Z, weights, p, circuit_fn=vqc, readout=lambda expval: (1 - expval) / 2, n_wires=N_QUBITS, chunk=2000):
    """Fraud scores of the transactions Z when circuit_fn runs under depolarizing noise p."""
    noisy_circuit = noisy_qnode(circuit_fn, p, n_wires)
    X_t = torch.tensor(Z)
    with torch.no_grad():
        return torch.cat([readout(noisy_circuit(X_t[k:k + chunk], weights)) for k in range(0, len(X_t), chunk)]).numpy()


def noise_locations(circuit_fn, *args, n_wires=N_QUBITS):
    """Number of depolarizing channels the noise model inserts: one per qubit of every native gate."""
    qnode = qml.transforms.decompose(qml.QNode(circuit_fn, qml.device("default.qubit", wires=n_wires)), gate_set=NATIVE_GATES)
    return sum(size * count for size, count in qml.specs(qnode)(*args)["resources"].gate_sizes.items())


def evaluate_under_noise(circuit_fn, weights, Z_valid, Z_test_part, y_valid=y_val, y_test_part=y_test,
                         readout=lambda expval: (1 - expval) / 2, noise_levels=NOISE_LEVELS, n_wires=N_QUBITS):
    clean = qml.QNode(circuit_fn, qml.device("default.qubit", wires=n_wires), interface="torch")
    with torch.no_grad():
        clean_val = readout(clean(torch.tensor(Z_valid), weights)).numpy()
        clean_test = readout(clean(torch.tensor(Z_test_part), weights)).numpy()
    rows = []
    for p in [0.0, *noise_levels]:
        if p == 0.0:
            val_scores, test_scores = clean_val, clean_test
        else:
            val_scores = noisy_predict(Z_valid, weights, p, circuit_fn, readout, n_wires)
            test_scores = noisy_predict(Z_test_part, weights, p, circuit_fn, readout, n_wires)
        threshold = choose_threshold(y_valid, val_scores)
        rows.append({"p": p, **evaluate(y_test_part, test_scores, threshold), "Spearman rho": spearmanr(clean_test, test_scores).statistic, "threshold": threshold})
    return pd.DataFrame(rows).set_index("p")

In [ ]:
# ✅ Checks din notebook (4.2, 9.1, 6.1, 6.2, 10.1) și numărul de porți / locații de zgomot
X_bal_check, y_bal_check = undersample(X_train, y_train, SPLIT_SEED)   # 4.2
assert len(y_bal_check) == 2 * y_train.sum() and y_bal_check.sum() == y_train.sum() and X_bal_check.index.is_unique
assert set(X_bal_check.index) <= set(X_train.index)

rng_check = np.random.default_rng(3)   # 9.1
y_c = rng_check.integers(0, 2, 300)
s_c = np.clip(0.3 * y_c + rng_check.random(300), 0, 1)
best_f1 = max(f1_score(y_c, (s_c >= t).astype(int)) for t in np.unique(s_c))
assert np.isclose(f1_score(y_c, (s_c >= choose_threshold(y_c, s_c)).astype(int)), best_f1)

dev4 = qml.device("default.qubit", wires=N_QUBITS)


def state_of(circuit_fn, *args):
    def full():
        circuit_fn(*args)
        return qml.state()
    return qml.QNode(full, dev4)()


def gate_counts(circuit_fn, *args):
    def full():
        circuit_fn(*args)
        return qml.state()
    return qml.specs(qml.transforms.decompose(qml.QNode(full, dev4), gate_set=NATIVE_GATES))()["resources"].gate_types


def feature_map_reference(x):
    for _ in range(FEATURE_MAP_REPS):
        for i in range(N_QUBITS):
            qml.Hadamard(wires=i)
            qml.RZ(2 * x[i], wires=i)
        for i in range(N_QUBITS):
            for j in range(i + 1, N_QUBITS):
                qml.IsingZZ(2 * (np.pi - x[i]) * (np.pi - x[j]), wires=[i, j])


Z_check = AngleEncoder(N_QUBITS, SPLIT_SEED).fit(X_bal_check).transform(X_bal_check)
for x_check in Z_check[:3]:   # 6.1
    assert np.isclose(abs(np.vdot(state_of(feature_map_reference, x_check), state_of(feature_map, x_check))) ** 2, 1.0)

w_check = torch.linspace(0, 3, (ANSATZ_REPS + 1) * N_QUBITS * 2, dtype=torch.float64).reshape(ANSATZ_REPS + 1, N_QUBITS, 2)   # 6.2
probabilities = np.abs(qml.math.to_numpy(state_of(ansatz, w_check))) ** 2
assert np.allclose(probabilities[[0, 5, 10, 15]], [0.049016, 0.01134, 0.037841, 0.048166], atol=1e-5)


@qml.qnode(qml.device("default.mixed", wires=1))   # 10.1: canalul depolarizant scalează <Z> cu (1 - 4p/3)
def noisy_ry(theta, p):
    qml.RY(theta, wires=0)
    qml.DepolarizingChannel(p, wires=0)
    return qml.expval(qml.PauliZ(0))


for theta, p in [(0.3, 0.0), (1.0, 0.1), (2.5, 0.3), (0.7, 0.75)]:
    assert np.isclose(noisy_ry(theta, p), (1 - 4 * p / 3) * np.cos(theta))

x_example, w_example = torch.tensor(Z_check[0]), init_weights(SEEDS[0])
full_counts = gate_counts(vqc, x_example, w_example)
N_GATES, N_CNOTS = sum(full_counts.values()), full_counts["CNOT"]
N_NOISE_LOCATIONS = noise_locations(vqc, x_example, w_example)
N_PARAMS = w_example.numel()
EXECUTIONS_PER_SAMPLE = 1 + 2 * N_PARAMS
assert (N_GATES, N_CNOTS, N_PARAMS, EXECUTIONS_PER_SAMPLE) == (93, 33, 32, 65), dict(full_counts)
assert N_NOISE_LOCATIONS == 93 + 33, "fiecare poartă primește câte un canal pe fiecare qubit atins: 93 + 33 = 126"
print(f"VQC: {N_GATES} porți native ({N_CNOTS} CNOT), {N_NOISE_LOCATIONS} canale depolarizante inserate, {N_PARAMS} parametri")
print("✅ Checks 4.2, 9.1, 6.1, 6.2, 10.1: trecute")

## 4. Ponderile antrenate: reproducere din Experimentul 1 și salvare

Pentru fiecare seed: `undersample(X_train, y_train, seed)` → `AngleEncoder(4, seed)` fitat pe acest set 1:1 → `init_weights(seed)` → `train_vqc_run`. Este exact procedura rulării 1:1 din Experimentul 1 (acolo setul 1:1 era tot ieșirea lui `undersample`, iar encoderul era fitat tot pe el).

**Verificare față de Experimentul 1** (`results/result_exp1/test1_full_without_noise/`, doar citit): loss-ul și AUPRC-ul pe validation la fiecare din cele 30 de epoci, epoca aleasă și AUPRC-ul final trebuie să coincidă. Dacă nu coincid, celula se oprește și **nu** salvează nimic: modelul reprodus nu ar fi același cu cel raportat.

Ponderile verificate se salvează în `results/result_exp2/weights/seed_<s>.pt` (ponderile epocii alese, epoca, AUPRC-ul și istoricul). La rulările următoare se încarcă și se verifică din nou, fără reantrenare.

In [ ]:
def load_jsonl(path, required=False):
    """Înregistrările din path. Un fișier obligatoriu lipsă oprește execuția, în loc să întoarcă o listă goală."""
    if not path.exists():
        if required:
            raise FileNotFoundError(f"fișier obligatoriu lipsă: {path}")
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def to_builtin(value):
    if isinstance(value, dict):
        return {k: to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, np.generic):
        return value.item()
    return value


def append_jsonl(path, record):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(to_builtin(record), ensure_ascii=False) + "\n")


exp1_runs = {r["seed"]: r for r in load_jsonl(EXP1_DIR / "runs.jsonl", required=True) if r["ratio"] == 1}
exp1_hist = {r["seed"]: r["history"] for r in load_jsonl(EXP1_DIR / "histories.jsonl", required=True) if r["ratio"] == 1}
assert set(SEEDS) <= set(exp1_runs) and set(SEEDS) <= set(exp1_hist), "lipsesc rulările 1:1 ale Experimentului 1"
exp1_config = json.loads((EXP1_DIR / "config.json").read_text(encoding="utf-8"))
assert (exp1_config["epochs"], exp1_config["batch_size"], exp1_config["learning_rate"], exp1_config["n_params"]) == (EPOCHS, BATCH_SIZE, LEARNING_RATE, 32)


def check_against_exp1(seed, best_epoch, best_val_ap, history):
    ref, ref_hist = exp1_runs[seed], exp1_hist[seed]
    assert len(history) == len(ref_hist) == EPOCHS, f"seed {seed}: număr diferit de epoci"
    for col in ["train loss", "validation AUPRC"]:
        a, b = np.array([h[col] for h in history]), np.array([h[col] for h in ref_hist])
        assert np.allclose(a, b, rtol=0, atol=1e-8), f"seed {seed}: '{col}' diferă de Experimentul 1 (diferență maximă {np.abs(a - b).max():.2e})"
    assert best_epoch == ref["epoca aleasă"], f"seed {seed}: epoca aleasă {best_epoch}, în Experimentul 1 {ref['epoca aleasă']}"
    assert np.isclose(best_val_ap, ref["val AUPRC"], rtol=0, atol=1e-9), f"seed {seed}: AUPRC {best_val_ap:.6f} vs {ref['val AUPRC']:.6f}"


WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
models, retrain_time = {}, 0.0
for seed in SEEDS:
    X_s, y_s = undersample(X_train, y_train, seed)
    encoder = AngleEncoder(N_QUBITS, seed).fit(X_s)
    Z_tr, Z_val = encoder.transform(X_s), encoder.transform(X_val)
    path = WEIGHTS_DIR / f"seed_{seed}.pt"
    if path.exists():
        saved = torch.load(path, weights_only=True)
        source = "încărcat"
    elif RETRAIN_IF_MISSING:
        run = train_vqc_run(Z_tr, y_s, Z_val, y_val, init_weights(seed), seed)
        saved = {"weights": run["weights"], "best_epoch": run["best_epoch"], "best_val_ap": run["best_val_ap"], "history": run["history"]}
        check_against_exp1(seed, saved["best_epoch"], saved["best_val_ap"], saved["history"])   # înainte de salvare
        torch.save(saved, path)
        retrain_time += run["time_train_s"]
        source = f"reprodus și salvat ({run['time_train_s']:.0f} s)"
    else:
        raise FileNotFoundError(f"{path} lipsește și RETRAIN_IF_MISSING = False")
    check_against_exp1(seed, saved["best_epoch"], saved["best_val_ap"], saved["history"])
    assert np.isclose(average_precision_score(y_val, predict(Z_val, saved["weights"])), saved["best_val_ap"])
    models[seed] = {"weights": saved["weights"], "encoder": encoder, "Z_val": Z_val, "best_epoch": saved["best_epoch"]}
    print(f"seed {seed}: {source}   epoca aleasă {saved['best_epoch']:2d}   val AUPRC (p = 0) {saved['best_val_ap']:.4f}")
print("✅ ponderi: identice cu modelele 1:1 din Experimentul 1 (istoric pe 30 de epoci, epoca aleasă, AUPRC)")

In [ ]:
# ✅ Sanity (ca în Partea 10.3): simulatorul cu matrici de densitate la p = 0 dă aceleași scoruri ca state-vector
w0, Zv0 = models[SEEDS[0]]["weights"], models[SEEDS[0]]["Z_val"]
sanity = np.abs(noisy_predict(Zv0[:2000], w0, 0.0) - predict(Zv0[:2000], w0)).max()
assert sanity < 1e-6, sanity   # precizie numerică între cele două simulatoare (~1e-7), neglijabilă pentru scoruri în [0, 1]
print(f"default.mixed la p = 0 vs default.qubit, primele 2000 de tranzacții din validation: diferență maximă {sanity:.1e}")

## 5. Etapa 1: evaluare sub zgomot pe validation

Pentru fiecare seed și fiecare p ∈ {0, 0.001, 0.01, 0.05}, aceeași logică ca `evaluate_under_noise`, dar **doar pe validation**:
1. scorurile validării: fără zgomot cu `predict` (state-vector), cu zgomot cu `noisy_predict` (modelul oficial de zgomot);
2. pragul = maximul F1 pe validation **la același p** (`choose_threshold`);
3. AUPRC, AUROC, precision, recall, F1, MCC la acest prag (`evaluate`), plus Spearman ρ între scorurile zgomotoase și cele fără zgomot ale validării.

**Atenție la citire:** pragul este ales și metricile sunt calculate pe **același** set (validation), deci precision / recall / F1 / MCC sunt optimiste. AUPRC, AUROC și ρ nu depind de prag. Valorile finale, raportabile, vin din etapa 2 pe test.

**Durată:** 3 niveluri × 5 seeds = 15 treceri zgomotoase peste 56.962 de tranzacții; notebookul oficial estimează 0,5–1 min per trecere pe laptop, deci ~8–15 min în total. Fiecare (seed, p) terminat se salvează imediat în `val_runs.jsonl`; o rulare întreruptă continuă de unde a rămas.

In [ ]:
config = {
    "experiment": EXPERIMENT_NAME, "seeds": SEEDS, "noise_levels": P_ALL,
    "model": "VQC original Task 1 (ZZ feature map x2 + EfficientSU2-style ansatz, 3 reps, 32 params, readout (1 - <Z0>)/2)",
    "training": "noise-free, reproduced from ratio_experiment_p0/full ratio 1:1 (verified per epoch), epochs %d" % EPOCHS,
    "balancing": "undersample(X_train, y_train, seed), ratio 1:1", "encoder": "AngleEncoder(4, seed) fitted on the 1:1 set",
    "noise_model": "decompose to NATIVE_GATES, DepolarizingChannel(p) after every gate on every wire (noisy_qnode), default.mixed",
    "threshold": "max F1 on validation scores at the same p (choose_threshold)",
    "stage1": "validation only: metrics and Spearman rho on validation", "stage2": "official evaluate_under_noise on test (RUN_TEST)",
    "split_seed": SPLIT_SEED, "split_source": split_source, "n_val": len(val_idx), "n_test": n_test_rows,
    "shots_per_execution": SHOTS_PER_EXECUTION, "data_source": data_source,
    "versions": {"pennylane": qml.__version__, "torch": torch.__version__, "sklearn": sklearn.__version__, "numpy": np.__version__},
}
CONFIG_PATH, VAL_RUNS_PATH, TEST_RUNS_PATH = RESULTS_DIR / "config.json", RESULTS_DIR / "val_runs.jsonl", RESULTS_DIR / "test_runs.jsonl"
if CONFIG_PATH.exists():
    previous = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    current = json.loads(json.dumps(to_builtin(config)))
    ignore = {"versions", "split_source"}   # split_source descrie de unde vine splitul (fișier sau reconstrucție), nu un parametru
    if {k: v for k, v in previous.items() if k not in ignore} != {k: v for k, v in current.items() if k not in ignore}:
        raise RuntimeError(f"{CONFIG_PATH} conține altă configurație. Nu suprascriu: schimbă EXPERIMENT_NAME sau mută directorul.")
else:
    CONFIG_PATH.write_text(json.dumps(to_builtin(config), indent=2, ensure_ascii=False), encoding="utf-8")

val_done = {(r["seed"], r["p"]) for r in load_jsonl(VAL_RUNS_PATH)}
n_todo = sum((s, p) not in val_done for s in SEEDS for p in NOISE_LEVELS)
print(f"treceri zgomotoase de făcut pe validation: {n_todo} din {len(SEEDS) * len(NOISE_LEVELS)}")

for seed in SEEDS:
    m = models[seed]
    clean_val = predict(m["Z_val"], m["weights"])
    for p in P_ALL:
        if (seed, p) in val_done:
            continue
        t0 = time.perf_counter()
        scores = clean_val if p == 0.0 else noisy_predict(m["Z_val"], m["weights"], p)
        elapsed = time.perf_counter() - t0
        threshold = choose_threshold(y_val, scores)
        metrics = evaluate(y_val, scores, threshold)
        if p == 0.0:   # la p = 0 trebuie să regăsim exact rularea 1:1 din Experimentul 1
            ref = exp1_runs[seed]
            assert np.isclose(metrics["AUPRC"], ref["val AUPRC"]) and np.isclose(threshold, ref["prag"]) and np.isclose(metrics["F1"], ref["F1"])
        record = {"seed": seed, "p": p, **metrics, "Spearman rho": spearmanr(clean_val, scores).statistic,
                  "threshold": float(threshold), "timp scorare [s]": elapsed}
        append_jsonl(VAL_RUNS_PATH, record)
        val_done.add((seed, p))
        print(f"seed {seed}  p = {p:<5}  val AUPRC {metrics['AUPRC']:.3f}  F1 {metrics['F1']:.3f}  rho {record['Spearman rho']:.4f}  prag {threshold:.3f}  ({elapsed:.0f} s)")

### Tabele și grafice (validation)

Medie ± abatere standard (ddof = 1) peste cele 5 seeds. În grafice, punctele gri sunt seeds individuale, iar marcajul albastru cu bare este media ± std.

In [ ]:
METRICS = ["AUPRC", "AUROC", "Spearman rho", "precision", "recall", "F1", "MCC"]


def load_results(path):
    records = [r for r in load_jsonl(path, required=True) if r["seed"] in SEEDS and r["p"] in P_ALL]
    missing = sorted({(s, p) for s in SEEDS for p in P_ALL} - {(r["seed"], r["p"]) for r in records})
    assert not missing, f"lipsesc rezultate (seed, p): {missing}"
    return pd.DataFrame(records).sort_values(["p", "seed"]).reset_index(drop=True)


def mean_std(values, digits=3):
    values = np.asarray(values, dtype=float)
    return f"{values.mean():.{digits}f} ± {values.std(ddof=1):.{digits}f}"


def summary_table(results):
    rows = []
    for p in P_ALL:
        part = results[results["p"] == p]
        rows.append({"p": p, **{m: mean_std(part[m], 4 if m == "Spearman rho" else 3) for m in METRICS}, "prag": mean_std(part["threshold"])})
    return pd.DataFrame(rows).set_index("p")


def plot_vs_p(results, label, path):
    xpos = np.arange(len(P_ALL))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, col, title in zip(axes, ["AUPRC", "F1", "Spearman rho"],
                              ["AUPRC vs p", "F1 la pragul max-F1 (ales la același p) vs p", "Spearman ρ, scoruri cu zgomot vs fără zgomot"]):
        groups = [results.loc[results["p"] == p, col].to_numpy(dtype=float) for p in P_ALL]
        for i, g in enumerate(groups):
            ax.scatter(np.full(len(g), xpos[i]), g, s=22, color=SEED_DOT_COLOR, alpha=0.7, zorder=2)
        means, stds = np.array([g.mean() for g in groups]), np.array([g.std(ddof=1) for g in groups])
        ax.errorbar(xpos, means, yerr=stds, color=LEGIT_COLOR, marker="o", markersize=8, linewidth=2, capsize=4, zorder=3)
        ax.set_xticks(xpos, ["0", *[str(p) for p in NOISE_LEVELS]])
        ax.set_xlabel("probabilitate depolarizantă p per poartă")
        ax.set_title(title)
    fig.suptitle(f"{label} · VQC original, 1:1, antrenat fără zgomot · medie ± std pe {len(SEEDS)} seeds (gri: seeds individuale)", fontsize=11)
    plt.tight_layout()
    fig.savefig(path, dpi=130)
    plt.show()


VAL_LABEL = "Etapa 1 — validation (prag și metrici pe același set)"
val_results = load_results(VAL_RUNS_PATH)
detail_cols = ["seed", "p", *METRICS, "threshold", "TP", "FP", "FN", "TN", "timp scorare [s]"]
print(f"Tabel detaliat — {VAL_LABEL}")
display(val_results[detail_cols].round(4))
val_results[detail_cols].to_csv(RESULTS_DIR / "val_detailed.csv", index=False)

val_summary = summary_table(val_results)
print(f"Rezumat, medie ± std pe {len(SEEDS)} seeds — {VAL_LABEL}")
display(val_summary)
val_summary.to_csv(RESULTS_DIR / "val_summary.csv")

plot_vs_p(val_results, VAL_LABEL, RESULTS_DIR / "val_noise_plots.png")

## 6. Cost ledger

Formula din Partea 8, cu $P = 32$ parametri și 1024 shots per execuție. Separăm două lucruri diferite:

Patru componente, separate, fiecare cu starea ei (executat / planificat):

1. **Antrenarea** (Experimentul 1, 1:1): 30 de epoci × 590 de exemple × 65 de execuții (forward + parameter shift).
2. **Validarea în timpul antrenării** (Experimentul 1): o trecere peste validation (56.962 de execuții) după fiecare epocă, 30 de treceri.
3. **Evaluarea pe validation, deja realizată.** (a) p = 0 cu ponderile alese: este **aceeași** trecere ca „validation finală” din Experimentul 1 și se numără **o singură dată**; în Exp. 2 scorurile au fost doar recalculate pe simulator. (b) câte o trecere pentru fiecare p = 0.001, 0.01, 0.05, executate în Exp. 2.
4. **Evaluarea pe test, încă neexecutată** (etapa 2): câte o trecere pentru fiecare din cele 4 niveluri.

Valorile din Experimentul 1 sunt citite din `runs.jsonl` și verificate față de formula din Partea 8. Pe un calculator cuantic real nu alegem p: zgomotul e cel al dispozitivului, deci o implementare reală plătește **un singur** nivel; cele 3 niveluri cu zgomot sunt scenarii simulate.

**Cost hardware ≠ timp de simulare.** Execuțiile și shots-urile (1024 per execuție) sunt nominale: simularea rulează cu `shots=None` și matrici de densitate exacte. Timpul efectiv al simulatorului este raportat într-un tabel separat; reproducerea ponderilor și recalcularea validării în etapa 2 de `evaluate_under_noise` sunt timp de simulator, nu cost hardware în plus.

In [ ]:
n_val = len(val_idx)
for s in SEEDS:   # costurile Exp. 1 trebuie să corespundă formulei din Partea 8
    assert exp1_runs[s]["exec training"] == EPOCHS * 590 * EXECUTIONS_PER_SAMPLE
    assert exp1_runs[s]["exec validation, toate epocile"] == EPOCHS * n_val
    assert exp1_runs[s]["exec validation finală"] == n_val
assert int((val_results["p"] > 0).sum()) == len(SEEDS) * len(NOISE_LEVELS), "lipsesc evaluări zgomotoase pe validation"
test_done = RUN_TEST and TEST_RUNS_PATH.exists()

# (componentă, fază, stare, execuții per trecere, treceri per seed); la antrenare o „trecere” este o epocă
ledger_rows = [
    ("1. antrenare: forward + parameter shift", "antrenare", "executat (Exp. 1)", 590 * EXECUTIONS_PER_SAMPLE, EPOCHS),
    ("2. validare după fiecare epocă, p = 0", "validare în timpul antrenării", "executat (Exp. 1)", n_val, EPOCHS),
    ("3a. validation p = 0, ponderile alese (= trecerea finală din Exp. 1, numărată o dată)", "evaluare validation", "executat (Exp. 1)", n_val, 1),
    ("3b. validation sub zgomot, p = 0.001 / 0.01 / 0.05", "evaluare validation", "executat (Exp. 2)", n_val, len(NOISE_LEVELS)),
    ("4. test, p = 0 / 0.001 / 0.01 / 0.05", "evaluare test", "executat (Exp. 2, etapa 2)" if test_done else "planificat (etapa 2, neexecutat)", n_test_rows, len(P_ALL)),
]
n_s = len(SEEDS)
ledger = pd.DataFrame([
    {"componentă": name, "fază": phase, "stare": status, "execuții / trecere": per_pass, "treceri / seed": passes,
     "execuții / seed": per_pass * passes, f"execuții, {n_s} seeds": per_pass * passes * n_s,
     f"shots, {n_s} seeds": per_pass * passes * n_s * SHOTS_PER_EXECUTION}
    for name, phase, status, per_pass, passes in ledger_rows
]).set_index("componentă")
totals = ledger.assign(stare=ledger["stare"].str.split(" ").str[0]).groupby("stare")[[f"execuții, {n_s} seeds", f"shots, {n_s} seeds"]].sum()

print(f"Cost estimat pe hardware (nominal, parameter shift, {SHOTS_PER_EXECUTION} shots/execuție), VQC baseline: "
      f"{N_GATES} porți, {N_CNOTS} CNOT, {N_NOISE_LOCATIONS} locații de zgomot")
display(ledger.map(lambda v: f"{v:,}" if isinstance(v, (int, np.integer)) else v))
print("Total executat separat de total planificat (nu se adună):")
display(totals.map(lambda v: f"{v:,}"))
ledger.to_csv(RESULTS_DIR / "cost_ledger.csv")

sim_time = pd.DataFrame([
    {"ce": "antrenarea 1:1 în Exp. 1 (5 seeds, din runs.jsonl)", "timp simulator [min]": sum(exp1_runs[s]["timp antrenare [s]"] for s in SEEDS) / 60},
    {"ce": "reproducerea ponderilor în această sesiune (0 dacă au fost încărcate)", "timp simulator [min]": retrain_time / 60},
    {"ce": "treceri zgomotoase pe validation (Exp. 2)", "timp simulator [min]": val_results.loc[val_results["p"] > 0, "timp scorare [s]"].sum() / 60},
]).set_index("ce")
print("Timp efectiv de simulare (informativ, depinde de mașină; NU este cost hardware):")
display(sim_time.round(1))
sim_time.to_csv(RESULTS_DIR / "simulator_time.csv")

## 7. Etapa 2: evaluarea finală pe test (protocolul oficial)

Se rulează **doar** cu `RUN_TEST = True` (în celula de configurație), după ce nu mai luăm decizii pe baza acestor rezultate. Pentru fiecare seed, encoderul acelui seed (fitat pe train) transformă testul, apoi `evaluate_under_noise` din notebookul oficial, neschimbat, calculează scorurile pe validation și test la fiecare p, alege pragul pe validation la același p și raportează metricile pe test și Spearman ρ pe test.

Verificare: pragurile alese de `evaluate_under_noise` trebuie să fie identice cu cele din etapa 1 (simularea e deterministă). Rezultatele se salvează în `test_runs.jsonl`, `test_summary.csv` și `test_noise_plots.png`. Durată: ~2 × 15 treceri zgomotoase (validation + test), adică ~15–30 min.

In [ ]:
if RUN_TEST:
    val_thresholds = val_results.set_index(["seed", "p"])["threshold"]
    tested = {r["seed"] for r in load_jsonl(TEST_RUNS_PATH)}
    for seed in SEEDS:
        if seed in tested:
            print(f"seed {seed}: deja evaluat pe test, sărit")
            continue
        m = models[seed]
        t0 = time.perf_counter()
        protocol = evaluate_under_noise(vqc, m["weights"], m["Z_val"], m["encoder"].transform(X_test))
        elapsed = time.perf_counter() - t0
        for p, row in protocol.iterrows():
            assert np.isclose(row["threshold"], val_thresholds[(seed, p)]), f"seed {seed}, p = {p}: pragul diferă de etapa 1"
        for p, row in protocol.iterrows():
            append_jsonl(TEST_RUNS_PATH, {"seed": seed, "p": float(p), **row.to_dict(), "timp protocol seed [s]": elapsed})
        print(f"seed {seed}: " + "   ".join(f"p = {p}: AUPRC {row['AUPRC']:.3f}" for p, row in protocol.iterrows()) + f"   ({elapsed / 60:.1f} min)")
else:
    print("RUN_TEST = False: testul nu a fost transformat și nu a fost scorat.")

In [ ]:
if RUN_TEST:
    TEST_LABEL = "Evaluare finală — test (prag ales pe validation la același p)"
    test_results = load_results(TEST_RUNS_PATH)
    print(f"Tabel detaliat — {TEST_LABEL}")
    display(test_results[["seed", "p", *METRICS, "threshold", "TP", "FP", "FN", "TN"]].round(4))
    test_results.to_csv(RESULTS_DIR / "test_detailed.csv", index=False)
    test_summary = summary_table(test_results)
    print(f"Baseline Task 2, medie ± std pe {len(SEEDS)} seeds — {TEST_LABEL}")
    display(test_summary)
    test_summary.to_csv(RESULTS_DIR / "test_summary.csv")
    plot_vs_p(test_results, TEST_LABEL, RESULTS_DIR / "test_noise_plots.png")

## 8. Cum citim rezultatele

- **AUPRC** (metrica principală, accent pe p = 0.01 și 0.05) și **AUROC** depind doar de ordinea scorurilor; **Spearman ρ** arată direct cât din această ordine supraviețuiește zgomotului. Le citim împreună: notebookul oficial arată că pe un singur seed AUPRC sau AUROC pot crește sub zgomot în timp ce ρ scade.
- **Precision, recall, F1, MCC** depind de prag. Pragul e ales din nou la fiecare p, deci o scădere aici înseamnă că nici pragul optim nu mai separă clasele la fel de bine, nu doar că scorurile s-au deplasat spre 0,5.
- Validation are doar 99 de fraude: mutarea unei singure fraude în clasament schimbă AUPRC vizibil. O diferență între niveluri contează doar dacă e mare față de std-ul dintre seeds și apare la majoritatea seeds.
- Etapa 1 e pe validation, cu pragul ales pe același set: este pentru înțelegere și pentru deciziile următoare (de exemplu simplificarea circuitului), nu pentru raportare. Cifrele raportate sunt cele din etapa 2, pe test.
- Lipsește încă referința clasică (regresie logistică pe aceleași 4 unghiuri, per seed), obligatorie în Task 2.